In [4]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [6]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
151,"It's 1982, Two years after the Iranian Embassy...",negative
536,I first saw this movie when I was very little....,positive
894,I have been a Mario fan for as long as I can r...,positive
417,THE FEELING of the need to have someone play t...,positive
119,excellent drama. very dark. i have never seen ...,positive


In [8]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [9]:
df = normalize_text(df)
df.head()

,review,sentiment
151,two year iranian embassy siege involved dramat...,negative
536,first saw movie little born around age s carto...,positive
894,mario fan long remember fond memory playing su...,positive
417,feeling need someone play role arbiter public ...,positive
119,excellent drama dark never seen california pho...,positive


In [10]:
df['sentiment'].value_counts()

sentiment
negative    269
positive    231
Name: count, dtype: int64

In [11]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [12]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
151,two year iranian embassy siege involved dramat...,0
536,first saw movie little born around age s carto...,1
894,mario fan long remember fond memory playing su...,1
417,feeling need someone play role arbiter public ...,1
119,excellent drama dark never seen california pho...,1


In [13]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [14]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [23]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/premmotetech1/MLOPS_CAPSTONE.mlflow')
dagshub.init(repo_owner='premmotetech1', repo_name='MLOPS_CAPSTONE', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")



❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\motep\anaconda3\envs\atlas\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=5c7f3f35-3d66-4498-a7be-0aef16bde822&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e86b3083fefc3e204da9575b90ebed7f4d65ccf26c0eaada7a4438b9b0192462




Accessing as premmotetech1

Initialized MLflow to track repo "premmotetech1/MLOPS_CAPSTONE"

Repository premmotetech1/MLOPS_CAPSTONE initialized!

2026/06/20 00:05:06 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1c5791cd94564ffab027292c5a1673ec', creation_time=1781894108262, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1781894108262, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [25]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-06-20 00:12:57,559 - INFO - Starting MLflow run...
2026-06-20 00:12:58,096 - INFO - Logging preprocessing parameters...
2026-06-20 00:12:59,050 - INFO - Initializing Logistic Regression model...
2026-06-20 00:12:59,066 - INFO - Fitting the model...
2026-06-20 00:12:59,093 - INFO - Model training complete.
2026-06-20 00:12:59,094 - INFO - Logging model parameters...
2026-06-20 00:12:59,397 - INFO - Making predictions...
2026-06-20 00:12:59,397 - INFO - Calculating evaluation metrics...
2026-06-20 00:12:59,415 - INFO - Logging evaluation metrics...
2026-06-20 00:13:00,696 - INFO - Saving and logging the model...
2026/06/20 00:13:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-06-20 00:13:18,807 - INFO - Model training and logging completed in 20.71 seconds.
2026-06-20 00:13:18,810 - INFO - Accuracy: 0.656
2026-06-20 00:13:18,811 - INFO - Precision: 0.6842105263157895
2026-06-20 00:13:18,812 - INFO - Recall: 0.609375
2026-06-20 00:13:18,

🏃 View run fortunate-sponge-549 at: https://dagshub.com/premmotetech1/MLOPS_CAPSTONE.mlflow/#/experiments/0/runs/7a49217167ad4a54adcd72b5fcfca62b
🧪 View experiment at: https://dagshub.com/premmotetech1/MLOPS_CAPSTONE.mlflow/#/experiments/0
